# Challenge 02 — Build a News Agent (Code-First)

In this challenge, you'll build a **News Agent** programmatically using the **Azure AI Foundry SDK v2** and Python. By the end, you'll be able to chat with your agent from this notebook.

## Learning Objectives
- Authenticate to Azure AI Foundry from code (using `DefaultAzureCredential` — no API keys)
- Create a versioned agent with custom system instructions
- Manage the conversation lifecycle: create agent → conversation → response
- Inspect and iterate on agent responses

## How This Notebook Works
- **Pre-filled cells** (✅): Environment setup, authentication, and validation — these are done for you.
- **Fill-in-the-blank cells** (🔨): Agent creation, conversation, and response — you complete the `___` placeholders.
- Each `___` has a `# TODO:` comment explaining what goes there.

## Key Concepts (v2 SDK)
- **`AIProjectClient`** — connects to your Foundry project
- **`create_version()`** — creates a versioned agent (not the older `create_agent()`)
- **Conversations API** — replaces the old threads/messages/runs pattern
- **`openai_client.responses.create()`** — invokes an agent against a conversation

---
## ✅ Setup — Install Dependencies

Run this cell once to install the required packages into your active environment.

In [1]:
# Install the core packages for this challenge
# This cell is complete — just run it.
%pip install azure-ai-projects azure-identity python-dotenv ipykernel --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## ✅ Imports & Environment Variables

This cell loads your `.env` file and reads the values you configured in Challenge 00. **Nothing to change here** — just run it and confirm the output shows your endpoint and model.

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import AzureCliCredential, InteractiveBrowserCredential, ChainedTokenCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

# Load environment variables from your .env file
load_dotenv(override=True)

# Read values from .env — these map to what you deployed in Challenge 00
PROJECT_ENDPOINT = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")
MODEL_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
TENANT_ID = os.getenv("AZURE_TENANT_ID")  # optional but recommended

# Validate that required values are present
assert PROJECT_ENDPOINT, "AZURE_AI_FOUNDRY_ENDPOINT is missing from .env"
assert MODEL_DEPLOYMENT_NAME, "AZURE_OPENAI_DEPLOYMENT_NAME is missing from .env"

print(f"✅ Endpoint: {PROJECT_ENDPOINT}")
print(f"✅ Model: {MODEL_DEPLOYMENT_NAME}")
print(f"✅ Tenant: {TENANT_ID or '(not set — will use CLI default)'}")

---
## ✅ Authenticate & Create the AIProjectClient

This cell authenticates using `ChainedTokenCredential` (tries Azure CLI first, falls back to browser login) and creates the `AIProjectClient` you'll use for the rest of the notebook. **Nothing to change here.**

In [3]:
# Authenticate — no API keys, just Azure Identity
credential = ChainedTokenCredential(
    AzureCliCredential(tenant_id=TENANT_ID) if TENANT_ID else AzureCliCredential(),
    InteractiveBrowserCredential(tenant_id=TENANT_ID) if TENANT_ID else InteractiveBrowserCredential(),
)

# Create the AIProjectClient connected to your Foundry project
project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
)
print("✅ Connected to Azure AI Foundry")

✅ Connected to Azure AI Foundry


---
## 🔨 Part 1: Create the News Agent

Now it's your turn. Create a versioned agent named `NewsAgent` with system instructions that define its persona as a travel news briefing assistant.

In the v2 SDK, agents are created as **versioned definitions** using `project_client.agents.create_version()` with a `PromptAgentDefinition`.

**Fill in the blanks below.** Each `___` has a hint in the comment above it.

In [ ]:
# TODO: Create your NewsAgent by filling in the blanks
#
# Requirements:
#   - agent_name: "NewsAgent"
#   - model: Use the model deployment name from your .env
#   - instructions: Define a persona that:
#       * Summarizes news topics in a concise, neutral, informative tone
#       * Organizes briefings with clear headlines and short summaries
#       * Covers multiple perspectives when discussing controversial topics
#       * Clarifies it does NOT have live news — uses training knowledge

news_agent = project_client.agents.create_version(
    agent_name="___",  # TODO: What should this agent be called?
    definition=PromptAgentDefinition(
        model=___,  # TODO: What variable holds your model deployment name?
        instructions=(
            "___"  # TODO: Write the system instructions for the agent
            # Hint: Be specific about tone, format, and limitations.
            # The more detail you give, the better the agent behaves.
        ),
    ),
)
print(f"✅ Agent created — ID: {news_agent.id}")
print(f"   Name: {news_agent.name}")
print(f"   Version: {news_agent.version}")

---
## 🔨 Part 2: Get an OpenAI Client & Create a Conversation

In the v2 SDK, agents communicate through **conversations** (replacing the old threads API). You get an OpenAI-compatible client from `project_client`, then create a conversation with an initial user message.

**Fill in the blanks** to create the OpenAI client and start a conversation.

In [ ]:
# TODO: Get an OpenAI client and create a conversation
#
# Hint: project_client has a method that returns an OpenAI-compatible client.
# Then use openai_client.conversations.create() with an items list.

openai_client = project_client.___  # TODO: Which method gets the OpenAI client?

user_message = "What's happening in Texas this week that a traveler should know?"

conversation = openai_client.conversations.create(
    items=[{"type": "message", "role": "___", "content": ___}],  # TODO: role? content variable?
)
print(f"✅ Conversation created — ID: {conversation.id}")

---
## 🔨 Part 3: Run the Agent Against the Conversation

Now invoke the agent to process the conversation and generate a response. In v2, you use `openai_client.responses.create()` with an `agent_reference` that points to your agent by name.

**Fill in the blanks** to get the agent's response.

In [ ]:
# TODO: Invoke the agent against the conversation
#
# Hint: openai_client.responses.create() takes:
#   - conversation: the conversation ID
#   - extra_body: a dict with an "agent_reference" containing the agent's name and type
#
# The agent_reference dict looks like:
#   {"name": <agent_name>, "type": "agent_reference"}

response = openai_client.responses.create(
    conversation=___,  # TODO: What's the conversation ID? (Hint: conversation.id)
    extra_body={"agent_reference": {"name": ___, "type": "agent_reference"}},  # TODO: agent name?
)
print(f"✅ Agent responded")
print(f"\n--- AGENT ---\n{response.output_text}")

---
## 🔨 Part 4: Multi-Turn Conversation

The conversation preserves history. Add a follow-up message to the existing conversation and invoke the agent again to see multi-turn behavior.

Use `openai_client.conversations.items.create()` to add a new message, then call `responses.create()` again.

**Fill in the blanks** — same pattern as Part 3.

In [ ]:
# TODO: Send a follow-up message and run the agent again
#
# This tests multi-turn conversation — the agent should remember
# the context from the first message.
#
# Hint: Use openai_client.conversations.items.create() to add a message,
# then openai_client.responses.create() to invoke the agent again.

second_message = "What are the key things happening in AI regulation right now?"

# Add the follow-up message to the existing conversation
openai_client.conversations.items.create(
    conversation_id=___,  # TODO: Which conversation?
    items=[{"type": "message", "role": "___", "content": ___}],  # TODO: role and content?
)
print(f"✅ Second message sent: '{second_message}'")

# Invoke the agent again on the same conversation
response2 = openai_client.responses.create(
    conversation=___,  # TODO: conversation ID?
    extra_body={"agent_reference": {"name": ___, "type": "agent_reference"}},  # TODO: agent name?
)
print(f"✅ Agent responded")
print(f"\n--- AGENT ---\n{response2.output_text}")

---
## 🔨 Part 5: Verify Your Agent in the Portal

Your agent now exists in your Foundry project. Before moving on:
1. Open the [Microsoft Foundry portal](https://ai.azure.com)
2. Navigate to your project
3. Find your `NewsAgent` in the agents list
4. Confirm it shows the model and instructions you configured

Run the cell below to print the details you should verify in the portal.

In [ ]:
# This cell is complete — just run it to see your agent details
print("=" * 50)
print("YOUR AGENT DETAILS (verify these in the portal):")
print("=" * 50)
print(f"  Agent ID:      {news_agent.id}")
print(f"  Agent Name:    {news_agent.name}")
print(f"  Agent Version: {news_agent.version}")
print(f"\n  Conversation:  {conversation.id}")
print("\n⚠️  DO NOT delete this agent — you need it for Challenge 03!")

---
## ⚠️ Cleanup (Only If Starting Over)

**Do NOT run this cell** unless you need to start over. Your `NewsAgent` is needed for Challenge 03 (Tools) and beyond.

In [ ]:
# ONLY run this if you need to start fresh — uncomment the lines below:

# openai_client.conversations.delete(conversation_id=conversation.id)
# print(f"Conversation deleted — ID: {conversation.id}")

# project_client.agents.delete_version(agent_name=news_agent.name, agent_version=news_agent.version)
# print(f"Agent deleted — name: {news_agent.name}, version: {news_agent.version}")

print("ℹ️ Agent preserved for Challenge 03.")